<a href="https://colab.research.google.com/github/phineas-pta/gg_colab_AI_playground/blob/main/trans_LLM.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# dịch = Gemma 4

cần phải chia file ra thảnh từng chương riêng, ví dụ dùng regex: `^(番外|序章|第([0-9]+|[零〇一二三四五六七八九十百千万两]+)\s?[章节回卷部篇])`

In [ ]:
%pip install -qU huggingface_hub
%pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

In [ ]:
!rm -f *.gguf
!hf download --local-dir=. mradermacher/gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic-i1-GGUF gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic.i1-Q4_K_M.gguf

In [ ]:
from llama_cpp import Llama

MODEL = Llama(
	model_path="./gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic.i1-Q4_K_M.gguf",
	n_gpu_layers=-1, seed=-1, use_mmap=False,
	# split_mode=llama_cpp.llama_split_mode.LLAMA_SPLIT_MODE_TENSOR, tensor_split=[.5]*2, # multi-gpu on kaggle
	flash_attn=True, swa_full=True, type_k=8, type_v=8, n_ctx=2**15, # should be 2**18
)

In [ ]:
SYSTEM_PROMPT = """Bạn là một dịch giả chuyên nghiệp, am hiểu thể loại tiểu thuyết tiên hiệp và kiếm hiệp Trung Quốc.

Nhiệm vụ của tôi (người dùng): Tôi sẽ cung cấp cho bạn văn bản gốc tiếng Trung của một chương truyện. Văn bản này có thể bao gồm tựa đề chương và các đoạn văn trong chương.

Nhiệm vụ của bạn (AI): Dịch các phần văn bản tiếng Trung được cung cấp sang tiếng Việt, tuân thủ nghiêm ngặt các nguyên tắc sau:
- Phong cách: Duy trì phong cách hành văn cổ điển, đặc trưng của thể loại tiên hiệp, kiếm hiệp, sử dụng ngôn ngữ giàu hình ảnh, giữ nguyên nhịp điệu, tiết tấu câu văn gốc để lột tả bầu không khí hào hùng, bi tráng.
- Ngôn ngữ: Ưu tiên sử dụng từ Hán Việt phổ biến, dễ hiểu với độc giả Việt Nam. Lựa chọn từ ngữ phù hợp với tính cách, địa vị của từng nhân vật.
- Danh từ riêng: Giữ nguyên tất cả các danh từ riêng (tên người, tên địa điểm, tên công pháp, tên môn phái, ...) ở dạng chữ Hán gốc, chỉ dịch sang phiên âm Hán Việt tương ứng.
- Chủ động và thông minh: Nếu gặp thuật ngữ khó hoặc mơ hồ, hãy cố gắng đưa ra bản dịch tốt nhất dựa trên ngữ cảnh và kiến thức về tiên hiệp. Nếu một cái tên rõ ràng là phiên âm (như tên một số yêu thú đặc biệt không có nghĩa Hán Việt rõ ràng), bạn có thể giữ nguyên hoặc đề xuất phiên âm thuần Việt.

Mục tiêu: Tạo ra một bản dịch chất lượng cao, truyền tải trọn vẹn tinh thần và phong cách của tác phẩm gốc, đảm bảo bản dịch bám sát nguyên tác về nội dung, đồng thời truyền tải được phong cách và giọng văn đặc trưng của thể loại, mang đến cho độc giả Việt Nam trải nghiệm đọc tốt nhất.

Chỉ trả lời bản dịch tiếng Việt, bao gồm cả tựa đề chương truyện nếu có trong văn bản. Không thêm bất kỳ lời bình luận, giải thích nào khác ngoài yêu cầu."""

def dịch(text, stream=True):
	response = MODEL.create_chat_completion(
		messages=[
			{"role": "system", "content": SYSTEM_PROMPT},
			{"role": "user", "content": text.strip().replace("\u3000", "").replace("\n\n", "\n")}
		],
		temperature=.3, top_p=.95, top_k=64, min_p=0., repeat_penalty=1.05, stream=stream
	)
	if stream:
		contents = []
		for chunk in response:
			if "content" in (c := chunk["choices"][0]["delta"]):
				print(c["content"], end="", flush=True)
				contents.append(c["content"])
		res = "".join(contents)
	else:
		res = response["choices"][0]["message"]["content"].replace("\n\n", "\n")
	return res.replace("\n\n", "\n")

In [ ]:
print("\n")
_ = dịch("""第一章 裂开
　　东城，洛府门前。
　　杨是非深呼吸一口气，有些紧张。
　　他放下肩头行囊，整理好还算干净的衣袍，又摸了摸自己的脸。
　　很好，不丑、且干净。
　　他再从行囊里取出一叠红纸，再三确认了记录的地址无误。
　　在红纸隙间，依稀可见有‘婚书’二字。
　　杨是非抬头看着大院门外挂着的‘洛府’牌匾，心生感慨。
　　“没想到，我穿越后也有当赘婿的机会。”
　　他的心思不由得回到半年前。
　　当时自己还是个平平无奇的毕业生，愁着该找什么工作能养活自己，又该在哪定居，往后的日常开销又该如何规划，房租、水电、保险…………
　　走出校园后，生活的压力仿佛准点打卡上班，一起到场。
　　可就在整理好心情，准备去早就联系好的公司第二轮面试之际——
　　看手机没留神一脚踩空，摔进了坑里。
　　好消息是，这坑不深。
　　可能是挖路在修某些管道，里面还有维修人员半蹲着。就算一头栽进去，顶多就是摔个满嘴泥。
　　坏消息是，他就是在这时候穿越的。
　　杨是非还深刻记得，当时自己平衡失控，双手下意识护住手机和头，一脸惊恐地看着坑里的维修小哥。
　　而小哥也恰好回过头来，一脸震惊地仰头看着他。
　　“卧槽！”“卧槽咧？！”
　　两人只相互匆忙打了个招呼，成了在现代社会的最后一声道别。
　　下一刻，他出现在陌生的田地上空，直挺挺得摔了下去。
　　摔了个鼻青脸肿外加各种扭伤挫伤，疼得哼哼唧唧半天都没能爬起来，差点以为自己得交代在这里。
　　直到凑巧有农妇途径此地，好心将他扶回了家中。
　　杨是非当时头脑混乱，跟农妇一家确认了许久，才知道自己真的穿越了。
　　期间闹过不少误会和笑话，甚至因为自己身上的‘奇装异服’和各种‘疯言疯语’，差点被扭送官府，好说歹说才让对方勉强收留了自己。
　　经过一番波折，他才在偏僻山村内安心养起了伤。
　　这期间想过父母的事、也想过该如何回家，甚至也想过是不是自己磕到脑袋生了臆想，只要眼睛一闭一睁，就会满头纱布的出现在医院病房里。
　　老妈会像往常一样坐在旁边骂自己又走路看手机，然后咬牙切齿又有些心疼地指向旁边挂着的各种瓶瓶罐罐，说自己还得修养一段时间才能吃饭，之后想吃想喝什么，他们回家再去做，就算不插食管了也不能吃外卖。
　　而老爸则是沉默无言，刷刷手机，抬头看着自己，无奈摇头叹气。
　　可惜，他没有做梦。
　　杨是非躺在床上呆了两天，颓丧了三天。
　　他不能理解，其他穿越者是怎么做到狠心抛弃一切的。
　　但人是调节能力很强的生物。
　　在身上的伤有所好转后，他默默收拾好所有心情，忍痛下了床，走进农田试着帮忙，开始去适应这个世界的生活。
　　他不是孩子了，得学会接受。
　　世界变化很大，但自己也得继续走。
　　而现在手中这份略显老旧的婚书，则是这一个月来坚持帮忙务农的‘回报’。
　　“——牛大婶她们，是怎么勾搭上这种大户人家的？”
　　杨是非呆站在洛府门前久久未动，俊秀脸庞上满是复杂。
　　在牛家村养伤的这小半个月，他大概明白了这个世界的门道。
　　虽有‘梁国’之称，但此国历史和印象中的各朝各代都截然不同，甚至有武者、江湖、神兵利器之类的存在，显然还沾了点武侠要素。
　　若是其他穿越人士，兴许能早早开始闯荡江湖混出个美名。
　　可惜，他一个半只脚迈向九九六的毕业生，不说能唱能跳吧，至少也是个半身不遂，跑个体测都能累得到处乱吐。
　　尤其是舞了舞牛大婶家里祖传的兵器——差点因此手腕骨折——他暂时放弃了享受江湖儿女情长的滋味。
　　况且，在养好身体后得先帮牛大婶一家多种地赚些银两，以此偿还救命之恩。
　　但不料恩情没还多少，倒是牛大婶某天突然将这一纸婚书塞了过来，满脸的笑嘻嘻。
　　“杨小子啊，我们家里没男丁，要不你替我们去赴约？”
　　“这怎么能成！这洛家说的是要和你们牛家结亲，我这外人——”
　　“嘿你别说，俺们是牛、你是杨，都是一家人！”
　　“………………”
　　虽然没想到牛大婶还会说冷笑话，但杨是非想了想，还是半推半就的同意了。
　　他觉得这是个机会。
　　好歹是穿越了，哪怕没学到什么上乘神功纵横江湖，至少也得活用一番自己的现代知识，试着在江湖上闯荡几回。
　　至少，不能吃软饭。
　　农家饭也不行。
　　哪怕在江湖上没混出名头、甚至洛家也不承认这一纸婚书，他也能借此为跳板，在县城等地落脚找些文书记账之类的活计。
　　他想，自己学了那么多年屁用没有的数学，或许就要在这一刻发光发热…………好歹高考分数不算低。
　　虽然自己基本忘了个干净。
　　但，只要能多赚点银两，养活好自己，也能多照顾牛大婶一家，让她们往后不用再去过风吹日晒的苦日子。
　　对，不吃软饭。
　　可如果江湖、商业、官场皆失利，甚至连工作都找不到…………
　　咳，再说。
　　杨是非的行动能力一直不错。
　　所以在他定下决心后，立刻用上在村里到处做工帮农攒下的钱财，给自己办了两套‘新衣’。跟牛大婶一家告别后，坐着顺风马车，赶了几十里山路来到了东城，站在了此地。
　　叩叩叩——
　　杨是非斟酌许久，还是鼓起勇气，敲响了洛府的大门。
　　他一个母胎单身，别说谈女朋友了，连小姑娘的手都没拉过一次。
　　如今要拿着别人家的婚书冒名顶替，独自一人大老远跑来登门当‘赘婿’，确实很尴尬。
　　但想想自己无权无势，更没钱没房…………
　　忍了。
　　眼下只能不断温习当地的口语说辞、默念着早已准备好的腹稿，想尽可能先给洛府的人留下个好印象。
　　虽然他还未见过那位洛家大小姐，不知对方性情如何，但既然承牛大婶好意接了婚书，在明面上是得先好好表现。
　　“晚辈杨是非，应婚书前来拜访。不知…………嗯？”
　　杨是非敲门的动作一顿，看着‘嘎吱’一声缓缓敞开的院门，愣了愣。
　　门没关。
　　杨是非一脸古怪，仰头看了眼天色。
　　阴云渐笼、月色难明。他今日是赶着点才堪堪到了东城，没来得及吃个晚饭就赶来拜会。
　　这世道不少飞贼都学了武，听说都能飞檐走壁，修为高深者碎金裂石都不在话下，区区一道墙的确和装饰无异，但好歹也算是个门面。晚上八九点别说给院门上锁、连门捎都没带上，这洛家的下人是不是有点玩忽职守？
　　算了，和自己没什么关系。
　　杨是非硬着头皮推开大门，往洛府里头瞧了瞧。
　　月光稀疏，隐约能看见花园锦簇、假山水泊，俨然是一派大户人家的门面大院，兴许是王公贵族也说不定。
　　与他路上打听来的情报正好对上。
　　这洛府似是京城的大户人家、地位颇高。
　　而洛大小姐便是‘离家出走’的黄花闺女，在东城擅自定了居。
　　最初，不少街坊还以为这户大小姐很快就会被抓回去，但没想到一住就是两三年，期间平平安安的没起过丝毫风波，不时还能遇见洛府侍女在外采购，似在此长居久住，东城里的百姓对此也就慢慢淡忘了。
　　倒是那洛府大小姐鲜少外出，见过对方长相的人极少。只听街坊流言，是一位国色天香的大美人。
　　“…………人呢？”
　　杨是非往院子里探了探头，隐约在后院方向看见一点灯火，前院并没有人影。
　　难道真忘了锁门？
　　他站在原地清了清嗓子，提高嗓门再喊了一声。
　　“………………”
　　洛府后院依旧没有回应。
　　杨是非紧了紧衣襟，只觉得有点微冷。
　　他看着空无一人的偌大庭院，心中无奈，准备先去找一家旅店住上一晚，等明日一早再来重新拜见。
　　毕竟只是接了婚书，又不是真结婚了。不打招呼擅闯院宅，哪怕是上门来结亲的，免不了被指指点点。
　　杨是非将沉重的院门缓缓拉回，准备转身离开。
　　但在这时，一只纤白似玉的小手从院门内探出，竖在了两扇大门之间。
　　杨是非被吓了一跳，连忙抵门停住，差点将对方的手给夹了。
　　“姑娘？”
　　他将大门重新推开，瞧见一位少女俏生生站在门后。
　　此女身穿青瓷长裙、皓白衬衣，身段更是娇小玲珑，束腰缎带上一串银铃随风轻响，如同风吟。稚嫩如白玉般的俏脸却是清冷无波，唯有一双灵动美眸忽扇轻眨，似乎有些调皮。
　　看着年纪，大概十四五岁？
　　只是个子颇矮，还不到自己的胸口处。
　　杨是非暗想这或许就是洛家的侍女，定了定神，迅速道明来意。
　　“是你啊。”少女微微颔首，嗓音脆美如歌。
　　这让杨是非松了口气，没出误会。
　　“既然天色已暗，在下待明日再——”
　　“不进来？”
　　少女抬手打断了他的话，侧过身，笑吟吟地指了指内院：“她们，都在里面。”
　　杨是非怔了怔：“会不会打搅了府上诸位？”
　　“不晚。”少女微笑道：“正巧。“
　　“…………也好。”
　　杨是非想了想，将行囊重新背起：“我先去拜见一下洛大小姐。”
　　少女只是轻笑一声，推开院门让开了位置。
　　杨是非走进院子，正想再多问问，却见她指着内院：“直接过去。”
　　“行。”
　　得洛府侍女同意，杨是非也没扭捏，跟上对方脚步沿庭院小径一路走去。
　　“………………”
　　两人一路无言，安静得针落可闻。
　　杨是非看着侍女在前摇曳行进的纤细背影，心中暗暗感慨。
　　不知对方是否习武，这走起路来，还真是一点脚步声都没有。
　　待绕过几条弯弯绕绕的石子小路后，他很快来到了灯火摇曳的闺房门前。
　　眼见四下并无人影，其他房间也没有灯火，料想那洛大小姐应该就在此屋。
　　杨是非看向驻足停步的少女，指着房门。“姑娘，可否引荐？”
　　“她知道你会来。”
　　少女笑得有些令人不安：“开门就好。”
　　杨是非皱了皱眉，察觉到些许古怪。
　　不知是这位侍女小妹妹美得太过匪夷所思、还是对方的言辞和态度颇为微妙。
　　可傻站在女子闺房门前也不是个事，他沉默片刻，踏上门前石阶，正要将房门叩响。
　　但透过门缝，隐约看见屋内有一道倩影正背对着房门，在忽明忽暗的灯火映照下极显得妖娆妩媚。乌黑长发及腰轻荡，却依旧遮掩不住那傲人曲线。
　　两位窈窕侍女正站在两旁，似乎在为其梳理长发。
　　杨是非手一顿，有些尴尬。
　　不是说好了要见自己，怎么看起来还在梳妆打扮？
　　他从未谈过恋爱、更没摸清这个时代的名门女子有何癖好或忌讳，只知个大概的习俗礼节，一时不知该不该开口提醒屋内的女子。
　　正犹豫着是否要等对方打扮完了再开口，却见屋内两位侍女梳理头发的动作变得越来越…………奇怪。
　　杨是非眉头微抖，心中莫名，下意识眯起眼睛想先看个清楚。
　　就见两位侍女渐渐用力用十指攥住了那位女子的长发，朝着两边缓缓扯开。
　　“！”
　　杨是非呆了呆，差点以为是遇见了侍女欺负落魄大小姐的戏码…………就是所谓的扯头发？
　　这洛府什么情况？
　　但还来不及咂舌感叹，他就看到了更为惊心动魄的一幕。
　　被扯住头发的女子竟一声不吭、仿佛完全不知痛楚般安静端坐着。而随着侍女逐渐扯动，如瀑长发几乎被分割成左右两半，而此女的头顶竟像是被分离开的齿轮，显露出原本咬合在一起的锯齿状裂痕，如瓜果开瓢、更像是一朵娇颜花朵般徐徐绽放开来。
　　“………………”
　　杨是非满脸僵硬，后退一步，心跳极快。
　　什么瓜果花朵，这分明是整个脑袋被扯开花了！""")

nén các file chương thành zip r up lên colab

In [ ]:
!unzip -q raw.zip -d raw
!mkdir trans

###

from pathlib import Path
from tqdm import tqdm

INPUT_FOLDER = Path("raw")
OUTPUT_FOLDER = Path("trans")

for input_file in (pbar := tqdm(sorted(INPUT_FOLDER.glob("*.txt")))):
	pbar.set_description(f"Processing {input_file.name}")
	output_file = OUTPUT_FOLDER.joinpath(input_file.name)
	if not output_file.exists():
		content = input_file.read_text(encoding="utf-8").strip().replace("\u3000", "").replace("\n\n", "\n")
		output_file.write_text(dịch(content), encoding="utf-8")

###

!zip -FSr trans.zip ./trans